In [ ]:
import pandas as pd
import re
from openai import OpenAI
from google.colab import userdata

OPENAI_API_KEY = userdata.get('openai_api')
client = OpenAI(api_key=OPENAI_API_KEY)

# **Import data**

In [ ]:
df = pd.read_csv("/content/fulltext_filtered.csv")
# df.head()

# --- Step 1: Clean OSF links ---

In [ ]:
def clean_osf_links(osf_links_str):
    if not osf_links_str:
        return ""
    # Split multiple links if separated by spaces, commas, or pipes
    links = re.split(r"[ ,|]+", osf_links_str)
    # Keep only valid OSF links (https://osf.io/xxxx)
    clean_links = [link for link in links if re.match(r"https?://osf\.io/[a-z0-9]+/?$", link)]
    return " ".join(clean_links)


# --- Step 2: LLM function ---

In [ ]:
import json

def classify_prereg_with_links(text, DOI, osf_links):

    preregistration_tool = [
      {
          "type": "function",
          "function": {
              "name": "classify_preregistration",
              "description": "Determine if this text contains a TRUE preregistered study claim by the authors.",
              "parameters": {
                  "type": "object",
                  "properties": {
                      "DOI": {
                          "type": "string",
                          "description": "The DOI of the associated article."
                      },
                      "real_prereg": {
                          "type": "boolean",
                          "enum": [True, False],
                          "description": "Whether the text contains a TRUE preregistered study claim by the authors."
                      },
                      "studies": {
                        "type": "array",
                        "items": {
                          "type": "object",
                          "properties": {
                            "study_number": {"type": ["string", "null"]},
                            "has_link": {"type": "boolean"},
                            "preregistration_link": {"type": ["string", "null"]}
                        },
                        "required": ["study_number", "has_link", "preregistration_link"]
                    }
                }
                      "reasoning": {
                          "type": "string",
                          "description": "Very brief explanation of the decision."
                      }
                  },
                  "required": ["DOI", "real_prereg", "studies", "reasoning"]
              }
          }
      }
  ]




    prompt = f"""
Determine if this text contains a TRUE preregistered study claim by the authors.

Return ONLY valid JSON:

{{
  "DOI": "{DOI}",
  "real_prereg": true/false,
  "studies": [
    {{
      "study_number": "string or null",
      "has_link": true/false,
      "preregistration_link": "string or null"
    }}
  ],
  "reasoning": "short explanation"
}}

Rules:
- Only include studies that are explicitly preregistered
- If multiple studies are preregistered, include all
- study_number examples: "Study 1", "Experiment 2", or null if unclear
- For each study, check the following list of OSF links and assign the correct link if mentioned in the text:
  {osf_links if osf_links else "none"}
- Ignore citations to other papers
- Ignore general discussion of preregistration

Text:
{text}
"""
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}],
        #temperature=0, # For GPT-4 model only
        tools=preregistration_tool,
        tool_choice={"type": "function", "function": {"name": "classify_preregistration"}}
    )


    message = response.choices[0].message
    tool_args = json.loads(message.tool_calls[0].function.arguments)
    return tool_args


# --- Step 3: Subset rows ---

In [ ]:
df_sample = df.iloc[500:520].copy()

# --- Step 4: Clean OSF links ---

In [ ]:
df_sample["clean_osf_links"] = df_sample["osf_links"].apply(clean_osf_links)

# --- Step 5: Run LLM on the subsetted rows ---

In [ ]:
results = []

results = df_sample.apply(
    lambda row: classify_prereg_with_links(
        text=row["fulltext"],              # full text for better link assignment
        DOI=row["DOI"],
        osf_links=row["clean_osf_links"]  # cleaned OSF links
    ),
    axis=1
)

In [ ]:
results

,0
500,"{'DOI': '10.1037/xhp0001085', 'real_prereg': F..."
501,"{'DOI': '10.1037/pspa0000341', 'real_prereg': ..."
502,"{'DOI': '10.1037/xge0001381', 'real_prereg': T..."
503,"{'DOI': '10.1037/pspp0000458', 'real_prereg': ..."
504,"{'DOI': '10.1037/xge0001378', 'real_prereg': T..."
505,"{'DOI': '10.1037/xge0001367', 'real_prereg': T..."
506,"{'DOI': '10.1037/emo0001228', 'real_prereg': T..."
507,"{'DOI': '10.1037/xhp0001097', 'real_prereg': F..."
508,"{'DOI': '10.1037/xhp0001084', 'real_prereg': F..."
509,"{'DOI': '10.1037/xhp0001089', 'real_prereg': T..."


# --- Step 6: Display results ---

In [ ]:
for r in results.head(10):
    try:
        parsed = json.loads(r)  # only if LLM returned valid JSON
        display(parsed)
    except:
        print(r)  # fallback if JSON parsing fails

{'DOI': '10.1037/xhp0001085', 'real_prereg': False, 'studies': '[]', 'reasoning': 'The text explicitly states that the study was not preregistered and contains no preregistered study claims.'}
{'DOI': '10.1037/pspa0000341', 'real_prereg': True, 'studies': '[{"study_number":"Study 1","has_link":true,"preregistration_link":"https://aspredicted.org/ZIY_HEK"},{"study_number":"Study 2","has_link":true,"preregistration_link":"https://aspredicted.org/ELO_JAL"},{"study_number":"Study 3","has_link":true,"preregistration_link":"https://aspredicted.org/CWP_HRX"},{"study_number":"Study 4","has_link":true,"preregistration_link":"https://aspredicted.org/9W6_XY2"},{"study_number":"Study 5","has_link":true,"preregistration_link":"https://aspredicted.org/L2X_T83"},{"study_number":"Study 6","has_link":true,"preregistration_link":"https://aspredicted.org/EGG_ZXJ"},{"study_number":"Study 7","has_link":true,"preregistration_link":"https://aspredicted.org/XIG_SSG"},{"study_number":"Study 8","has_link":true,

## **Part 2. Identifying OSF Preregistration Tempaltes**
Now we want the LLM to take a look at those preregistrations and determine whether they match the OSF preregistration template.

# Step 1: Extract all OSF links from your LLM outputs



In [ ]:
import json
import re

osf_records = []

for r_item in results.values:
    try:
        # r_item is already a dictionary from the 'results' Series
        studies_data = r_item.get("studies") # Get the studies field

        if studies_data is None or studies_data == '': # Handle None or empty string directly
            studies_list = []
        elif isinstance(studies_data, str):
            try:
                studies_list = json.loads(studies_data)
            except json.JSONDecodeError:
                studies_list = [] # Fallback for malformed JSON string
        elif isinstance(studies_data, list): # Handle cases where 'studies' might already be a list
            studies_list = studies_data
        else:
            studies_list = [] # Default to empty for other unexpected types

        for study in studies_list:
            link = study.get("preregistration_link")
            if link:
                osf_records.append({
                    "DOI": r_item.get("DOI"),
                    "study_number": study.get("study_number"),
                    "link": link
                })

    except Exception as e:
        print(f"ERROR: {e} | RAW ITEM: {r_item}")
        continue

print(f"osf_records has {len(osf_records)} items")
print(osf_records[:3])

# Convert to dataframe
df_osf = pd.DataFrame(osf_records)

osf_records has 35 items
[{'DOI': '10.1037/pspa0000341', 'study_number': 'Study 1', 'link': 'https://aspredicted.org/ZIY_HEK'}, {'DOI': '10.1037/pspa0000341', 'study_number': 'Study 2', 'link': 'https://aspredicted.org/ELO_JAL'}, {'DOI': '10.1037/pspa0000341', 'study_number': 'Study 3', 'link': 'https://aspredicted.org/CWP_HRX'}]


# Step 2: Get OSF template using OSF's API

In [ ]:
import time
import re
import requests

def get_osf_registration_template(url):
    try:
        if "aspredicted.org" in url:
            return {"template": "AsPredicted", "status": "non_osf", "all_registrations": []}

        match = re.search(r"osf\.io/([a-z0-9]{5})", url, re.IGNORECASE)
        if not match:
            return {"template": None, "status": "could_not_parse_url", "all_registrations": []}

        node_id = match.group(1)
        headers = {"User-Agent": "Mozilla/5.0"}

        for attempt in range(3):
            try:
                # 1. Try as a registration directly
                r = requests.get(f"https://api.osf.io/v2/registrations/{node_id}/", headers=headers, timeout=30)
                if r.status_code == 200:
                    attrs = r.json()["data"]["attributes"]
                    template = attrs.get("registration_supplement", None)
                    return {
                        "template": template,
                        "status": "ok",
                        "all_registrations": [{
                            "url": f"https://osf.io/{node_id}",
                            "template": template,
                            "date": attrs.get("date_registered"),
                            "title": attrs.get("title")
                        }]
                    }

                # 2. Try as a project node
                r = requests.get(f"https://api.osf.io/v2/nodes/{node_id}/registrations/", headers=headers, timeout=30)
                if r.status_code == 200:
                    data = r.json().get("data", [])
                    if data:
                        data.sort(key=lambda x: x["attributes"]["date_registered"], reverse=True)
                        all_registrations = [
                            {
                                "url": f"https://osf.io/{d['id']}",
                                "template": d["attributes"].get("registration_supplement"),
                                "date": d["attributes"].get("date_registered"),
                                "title": d["attributes"].get("title")
                            }
                            for d in data
                        ]
                        # uses_osf_template is True if ANY registration used the OSF template
                        osf_templates = [r for r in all_registrations if r["template"] == "OSF Preregistration"]
                        top_template = all_registrations[0]["template"]  # most recent
                        return {
                            "template": top_template,
                            "status": "ok_via_node",
                            "all_registrations": all_registrations
                        }
                    else:
                        return {"template": None, "status": "node_has_no_registrations", "all_registrations": []}

                return {"template": None, "status": f"http_{r.status_code}", "all_registrations": []}

            except requests.exceptions.Timeout:
                if attempt == 2:
                    return {"template": None, "status": "timeout_after_3_attempts", "all_registrations": []}
                time.sleep(2)

    except Exception as e:
        return {"template": None, "status": f"error: {e}", "all_registrations": []}


# Run on all osf_records
osf_template_results = []

for i, row in df_osf.iterrows():
    url = row["link"] if row["link"].startswith("http") else "https://" + row["link"]
    result = get_osf_registration_template(url)
    osf_template_results.append({
        "DOI": row["DOI"],
        "study_number": row["study_number"],
        "link": row["link"],
        "template": result["template"],
        "uses_osf_template": result["template"] in ["OSF Preregistration", "Prereg Challenge"],
        "status": result["status"]
    })
    time.sleep(0.3)

df_osf_classified = pd.DataFrame(osf_template_results)
print(df_osf_classified["template"].value_counts())
df_osf_classified.head(10)

template
AsPredicted                                      13
OSF Preregistration                               8
Preregistration Template from AsPredicted.org     5
Open-Ended Registration                           4
Name: count, dtype: int64


,DOI,study_number,link,template,uses_osf_template,status
0,10.1037/pspa0000341,Study 1,https://aspredicted.org/ZIY_HEK,AsPredicted,False,non_osf
1,10.1037/pspa0000341,Study 2,https://aspredicted.org/ELO_JAL,AsPredicted,False,non_osf
2,10.1037/pspa0000341,Study 3,https://aspredicted.org/CWP_HRX,AsPredicted,False,non_osf
3,10.1037/pspa0000341,Study 4,https://aspredicted.org/9W6_XY2,AsPredicted,False,non_osf
4,10.1037/pspa0000341,Study 5,https://aspredicted.org/L2X_T83,AsPredicted,False,non_osf
5,10.1037/pspa0000341,Study 6,https://aspredicted.org/EGG_ZXJ,AsPredicted,False,non_osf
6,10.1037/pspa0000341,Study 7,https://aspredicted.org/XIG_SSG,AsPredicted,False,non_osf
7,10.1037/pspa0000341,Study 8,https://aspredicted.org/HXR_JHV,AsPredicted,False,non_osf
8,10.1037/xge0001381,Experiment 1,https://osf.io/ud5j8/,OSF Preregistration,True,ok
9,10.1037/xge0001381,Experiment 2,https://osf.io/6y325/,OSF Preregistration,True,ok


save dataframe

In [ ]:
df_osf_classified.to_csv("/content/osf_classified_GPT5-nano.csv", index=False) # Update with GPT model for comparison

In [ ]:
df[df["DOI"] == "10.1037/pspi0000424"].index.tolist()

[513]